# Estratégia de Reversão à Média — Parte 2: Clusterização e Treinamento

Este notebook cobre:
- Identificação de regimes de mercado com K-Means
- Treinamento dos modelos CatBoost (principal + meta-modelo) por cluster
- Avaliação e seleção do melhor modelo

## 1. Configuração do Ambiente

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

from labeling_lib import get_labels_filter
from tester_lib import tester

%matplotlib inline
print('Ambiente configurado com sucesso.')

## 2. Hiperparâmetros

> Mantenha os mesmos valores do notebook anterior para consistência.

In [ ]:
hyper_params = {
    'symbol':       'EURGBP_H1',
    'markup':       0.00010,
    'stop_loss':    0.02000,
    'take_profit':  0.00200,
    'periods':      [i for i in range(5, 300, 30)],
    'periods_meta': [10],
    'backward':     datetime(2000, 1, 1),
    'forward':      datetime(2021, 1, 1),
    'n_clusters':   10,
    'rolling':      200,
}

## 3. Carregamento e Preparação dos Dados

In [ ]:
def get_prices() -> pd.DataFrame:
    csv_path = os.path.join('..', hyper_params['symbol'] + '.csv')
    p = pd.read_csv(csv_path, sep=r'\s+')
    pFixed = pd.DataFrame(columns=['time', 'close'])
    pFixed['time'] = p['<DATE>'] + ' ' + p['<TIME>']
    pFixed['time'] = pd.to_datetime(pFixed['time'], format='mixed')
    pFixed['close'] = p['<CLOSE>']
    pFixed.set_index('time', inplace=True)
    pFixed.index = pd.to_datetime(pFixed.index, unit='s')
    return pFixed.dropna()

def get_features(data: pd.DataFrame) -> pd.DataFrame:
    pFixed = data.copy()
    pFixedC = data.copy()
    count = 0
    for i in hyper_params['periods']:
        pFixed[str(count)] = pFixedC.rolling(i).mean()
        count += 1
    for i in hyper_params['periods_meta']:
        pFixed[str(count) + 'meta_feature'] = pFixedC.rolling(i).skew()
        count += 1
    return pFixed.dropna()

dataset = get_features(get_prices())
print(f'Dataset pronto: {dataset.shape[0]:,} linhas × {dataset.shape[1]} colunas')

## 4. Clusterização — Identificação de Regimes de Mercado

O K-Means é aplicado sobre as features de **skewness** (assimetria) para dividir a série temporal em regimes com características estatísticas distintas.

- Um alto número de clusters captura mais nuances, mas pode levar a clusters com poucos exemplos
- O recomendado é entre **5 e 10 clusters**

In [ ]:
def clustering(data: pd.DataFrame, n_clusters: int) -> pd.DataFrame:
    # Usa apenas o período de treino para definir os clusters
    train_data = data[
        (data.index < hyper_params['forward']) &
        (data.index > hyper_params['backward'])
    ].copy()
    meta_X = train_data.loc[:, train_data.columns.str.contains('meta_feature')]
    train_data['clusters'] = KMeans(n_clusters=n_clusters, n_init='auto').fit(meta_X).labels_
    return train_data

clustered_full = clustering(dataset, n_clusters=hyper_params['n_clusters'])

print(f'Distribuição de candles por cluster (apenas treino):')
print(clustered_full['clusters'].value_counts().sort_index())

### Visualização dos regimes de mercado

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
scatter = ax.scatter(
    clustered_full.index,
    clustered_full['close'],
    c=clustered_full['clusters'],
    cmap='tab10',
    s=2,
    alpha=0.7
)
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_title('Regimes de Mercado identificados pelo K-Means (período de treino)')
ax.set_xlabel('Data')
ax.set_ylabel('Preço')
plt.tight_layout()
plt.show()

## 5. Funções de Treinamento e Avaliação

In [ ]:
def test_model(result: list, stop: float, take: float, show_plot: bool = False):
    """Avalia o modelo treinado no dataset completo (treino + validação OOS)."""
    pr_tst = get_features(get_prices())
    X = pr_tst[pr_tst.columns[1:]]
    X_meta = X.copy()
    X      = X.loc[:, ~X.columns.str.contains('meta_feature')]
    X_meta = X_meta.loc[:, X_meta.columns.str.contains('meta_feature')]

    pr_tst['labels']      = result[0].predict_proba(X)[:, 1]
    pr_tst['meta_labels'] = result[1].predict_proba(X_meta)[:, 1]
    pr_tst['labels']      = pr_tst['labels'].apply(lambda x: 0.0 if x < 0.5 else 1.0)
    pr_tst['meta_labels'] = pr_tst['meta_labels'].apply(lambda x: 0.0 if x < 0.5 else 1.0)

    return tester(
        pr_tst, stop, take,
        hyper_params['forward'], hyper_params['backward'],
        hyper_params['markup'], show_plot
    )

def fit_cluster_models(clustered_data: pd.DataFrame, meta_data: pd.DataFrame) -> list:
    """Treina o modelo principal e o meta-modelo para um cluster específico."""
    X      = clustered_data[clustered_data.columns[:-1]]
    X_meta = meta_data[meta_data.columns[:-1]]
    X      = X.loc[:, ~X.columns.str.contains('meta_feature')]
    X_meta = X_meta.loc[:, X_meta.columns.str.contains('meta_feature')]

    y      = clustered_data['labels'].astype('int16')
    y_meta = meta_data['clusters'].astype('int16')

    train_X,   test_X,   train_y,   test_y   = train_test_split(X,      y,      train_size=0.7, shuffle=True)
    train_X_m, test_X_m, train_y_m, test_y_m = train_test_split(X_meta, y_meta, train_size=0.7, shuffle=True)

    model = CatBoostClassifier(
        iterations=1000, eval_metric='Accuracy',
        verbose=False, use_best_model=False,
        task_type='CPU', thread_count=-1
    )
    model.fit(train_X, train_y, eval_set=(test_X, test_y), early_stopping_rounds=30)

    meta_model = CatBoostClassifier(
        iterations=500, eval_metric='F1',
        verbose=False, use_best_model=True,
        task_type='CPU', thread_count=-1
    )
    meta_model.fit(train_X_m, train_y_m, eval_set=(test_X_m, test_y_m), early_stopping_rounds=25)

    R2 = test_model([model, meta_model], hyper_params['stop_loss'], hyper_params['take_profit'])
    if math.isnan(R2):
        R2 = -1.0

    return [R2, model, meta_model]

print('Funções definidas.')

## 6. Loop de Treinamento

Para cada cluster identificado pelo K-Means:
1. Aplica o método de rotulagem escolhido
2. Treina o modelo principal e o meta-modelo
3. Avalia o R² no dataset completo

> **Método ativo**: altere a chamada de `get_labels_*` abaixo para experimentar outros métodos de rotulagem.
>
> Descomente o método desejado e comente o atual.

In [ ]:
models = []
data = clustering(dataset, n_clusters=hyper_params['n_clusters'])
sorted_clusters = sorted(data['clusters'].unique())

for clust in sorted_clusters:
    clustered_data = data[data['clusters'] == clust].copy()

    if len(clustered_data) < 500:
        print(f'Cluster {clust}: poucos exemplos ({len(clustered_data)}), pulando.')
        continue

    # ── Escolha o método de rotulagem ──────────────────────────────────────────
    clustered_data = get_labels_filter(
        clustered_data,
        rolling=hyper_params['rolling'],
        quantiles=[0.45, 0.55],
        polyorder=3
    )
    # from labeling_lib import get_labels_multiple_filters
    # clustered_data = get_labels_multiple_filters(clustered_data, rolling_periods=[50,100,200], quantiles=[.45,.55], window=100, polyorder=3)

    # from labeling_lib import get_labels_filter_bidirectional
    # clustered_data = get_labels_filter_bidirectional(clustered_data, rolling1=50, rolling2=200, quantiles=[.45,.55], polyorder=3)

    # from labeling_lib import get_labels_mean_reversion
    # clustered_data = get_labels_mean_reversion(clustered_data, markup=hyper_params['markup'], min_l=1, max_l=15, rolling=0.5, quantiles=[.45,.55], method='spline', shift=0)

    # from labeling_lib import get_labels_mean_reversion_multi
    # clustered_data = get_labels_mean_reversion_multi(clustered_data, markup=hyper_params['markup'], min_l=1, max_l=15, windows=[0.2,0.3,0.5], quantiles=[.45,.55])

    # from labeling_lib import get_labels_mean_reversion_v
    # clustered_data = get_labels_mean_reversion_v(clustered_data, markup=hyper_params['markup'], min_l=1, max_l=15, rolling=0.2, quantiles=[.45,.55], method='spline', shift=0, volatility_window=100)
    # ──────────────────────────────────────────────────────────────────────────

    if len(clustered_data) < 100:
        print(f'Cluster {clust}: exemplos insuficientes após rotulagem, pulando.')
        continue

    print(f'Cluster {clust}: {len(clustered_data)} exemplos rotulados. Treinando...')
    clustered_data = clustered_data.drop(['close', 'clusters'], axis=1)

    meta_data = data.copy()
    meta_data['clusters'] = meta_data['clusters'].apply(lambda x: 1 if x == clust else 0)

    result = fit_cluster_models(clustered_data, meta_data.drop(['close'], axis=1))
    print(f'  → R²: {result[0]:.4f}')
    models.append(result)

print(f'\nTotal de modelos treinados: {len(models)}')

## 7. Seleção e Avaliação do Melhor Modelo

In [ ]:
# Ordena pelo R² e seleciona o melhor
models.sort(key=lambda x: x[0])
best_model = models[-1]

print(f'Melhor R²: {best_model[0]:.4f}')

# Exibe o gráfico de desempenho
R2 = test_model(best_model[1:], hyper_params['stop_loss'], hyper_params['take_profit'], show_plot=True)
print(f'R² final (com gráfico): {R2:.4f}')

### Pontuação do melhor modelo no conjunto de validação

In [ ]:
print('Melhor score de validação do modelo principal:')
print(best_model[1].get_best_score())

### Ranking de todos os modelos treinados

In [ ]:
ranking = pd.DataFrame(
    [{'Posição': i+1, 'R²': round(m[0], 4)} for i, m in enumerate(reversed(models))]
)
ranking

---

## Próximo Passo

Continue no notebook **`03_Exportacao_ONNX.ipynb`** para exportar o melhor modelo para o formato ONNX e integrá-lo ao MetaTrader 5.

> **Importante:** o objeto `best_model` precisa ser passado para o próximo notebook. Você pode salvar com `pickle` ou simplesmente executar todos os notebooks em sequência na mesma sessão.

In [ ]:
# Opcional: salvar o melhor modelo em disco para uso no próximo notebook
import pickle

save_path = os.path.join('..', 'best_model.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(best_model, f)

print(f'Modelo salvo em: {os.path.abspath(save_path)}')